In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math
from pydantic import BaseModel, Field, model_validator 


In [ ]:
class GPT3Config(BaseModel):
    vocab_size: int = Field(default=50257, gt=0, description="Vocabulary size") # 256 is the base token + 50000 BPE (Byte Pair Encoding) + 1 special token (end-of-text token) = 50257
    context_length: int = Field(default=1024, gt=0, description="Max context length/Block size")
    hidden_size: int = Field(default=768, gt=0, description="model/hidden dimension aka d_model")  # embed_dim == hidden_size == n_embed == d_model
    n_layer: int = Field(default=12, gt=0, description="number of stacked decoder blocks")
    num_head: int = Field(default=12, gt=0, description="number of attention heads")
    intermediate_step: int = Field(default=3072, gt=0, description="FFN inner dimension")
    dropout: float = Field(default=0.1, ge=0.0, description="Residual dropout")
    attention_dropout: float = Field(default=0.1, ge=0.0, description="attention dropout")
    embedding_dropout: float = Field(default=0.1, ge=0.0, description="embedding dropout")

model_config = {"frozen": True}

@model_validator(mode = "after")
def check_head_division(self):
    if self.hidden_size % self.num_head != 0:
        raise ValueError(
            f"hidden_size ({self.hidden_size}) must be devisible by num_head ({self.num_head})"
            f"got remainder {self.hidden_size % self.num_head}"
        )
    return self 


In [ ]:
class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.num_head = config.num_head
        self.hidden_size = config.hidden_size 
        self.head_dim = self.hidden_size // self.num_head 

        self.qkv_proj = nn.Linear(config.hidden_size, 3 * config.hidden_size)
        self.out_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.out_proj.RESIDUAL_SCALE_INIT = True  # control signal variance and prevent vanishing or exploding gradient

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(torch.ones(config.context_length, config.context_length))

        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k , v = qkv.split(C, dim = 2)

        q = q.view(B, T, self.num_head, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_head, self.head_dim).transpose(1,2)

        attention = q @ k.tranpose(-2, -1) / math.sqrt(self.head_dim) # [B, H, T, D] @ [B. H, D, T] --> [B, H, T, T]
        attention = attention.masked_fill(self.causal_mask[:, :, T, T] == 0, float("-inf"))
        attention = self.attention_dropout(F.softmax(attention, dim = -1))

        score = attention @ v # [B, H, T, T] @ [B, H, T, D ] --> [B, H, T, D]
        score = score.transpose(1, 2).contiguous().view(B, T, C)

        return self.residual_dropout(self.out_proj(score))


In [ ]:
class PositionWiseFNN(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_step) 
        self.fc2 = nn.Linear(config.intermediate_step, config.hidden_size)

        self.gelu = nn.GELU(approximate= "tanh" )
        self.dropout = nn.Dropoout(config.dropout)

        self.fc2.RESIDUAL_SCALE_INIT = True 

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.hidden_size)
        self.attention = MaskedMultiHeadSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.hidden_size)
        self.ffn = PositionWiseFNN(config)


    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
